In [29]:
import pandas as pd
import numpy as np
import joblib
import json
import os
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error


In [30]:
BASE_DIR = Path(os.getcwd())
print( BASE_DIR)

/Users/salioudunn/Desktop/BootCamp/projects/MarketIQ/DataScience


In [31]:
LOOKBACK = 3
ROLLING_WINDOW = 5
EXPORT_DIR = "export"
os.makedirs(EXPORT_DIR, exist_ok=True)

TICKERS = {
    "GOOGL": {
        "model_path": BASE_DIR / "Code" / "Historical_Model" / "Google" / "export" / "google_xgb_model.pkl",
        "csv_path": BASE_DIR / "Yahoo_Finance" / "Google" / "GOOG_historical_stock.csv",
    },
    "AAPL": {
        "model_path": BASE_DIR / "Code" / "Historical_Model" / "Apple" / "export" / "apple_xgb_model.pkl",
        "csv_path": BASE_DIR / "Yahoo_Finance" / "Apple" / "AAPL_historical_stock.csv",
    },
    "MSFT": {
        "model_path": BASE_DIR / "Code" / "Historical_Model" / "Microsoft" / "export" / "microsoft_xgb_model.pkl",
        "csv_path": BASE_DIR / "Yahoo_Finance" / "Microsoft" / "MSFT_historical_data.csv",
    },
}

FRED_FILES = {
    "interest_rate": BASE_DIR / "FRED" / "interest_rate.csv",
    "inflation_rate": BASE_DIR / "FRED" / "inflation.csv",
    "unemployment_rate": BASE_DIR / "FRED" / "unemployement_rate.csv",
    "gdp_growth": BASE_DIR / "FRED" / "gdp.csv",
}

FEATURE_COLUMNS = [
    "close_lag_1", "close_lag_2", "close_lag_3",
    "rolling_mean_5", "rolling_std_5",
    "interest_rate", "inflation_rate", "unemployment_rate", "gdp_growth",
]

print("FEATURE_COLUMNS:", FEATURE_COLUMNS)

FEATURE_COLUMNS: ['close_lag_1', 'close_lag_2', 'close_lag_3', 'rolling_mean_5', 'rolling_std_5', 'interest_rate', 'inflation_rate', 'unemployment_rate', 'gdp_growth']


In [32]:
for name, cfg in TICKERS.items():
    print(name, "model:", cfg["model_path"].exists(), "| csv:", cfg["csv_path"].exists())

for name, path in FRED_FILES.items():
    print(name, "->", path.exists())

GOOGL model: True | csv: True
AAPL model: True | csv: True
MSFT model: True | csv: True
interest_rate -> True
inflation_rate -> True
unemployment_rate -> True
gdp_growth -> True


In [33]:
def load_fred_csv(path, value_col_name):
    df = pd.read_csv(path)
    date_col = next(c for c in df.columns if c.lower() in ("date", "index", "unnamed: 0", "observation_date"))
    df = df.rename(columns={date_col: "date"})
    df["date"] = pd.to_datetime(df["date"])
    value_col = next(c for c in df.columns if c != "date")
    df = df.rename(columns={value_col: value_col_name})
    return df[["date", value_col_name]].sort_values("date").dropna()

In [34]:
test = load_fred_csv(FRED_FILES["interest_rate"], "interest_rate")
print(test.shape)
test.tail()

(26329, 2)


,date,interest_rate
26324,2026-07-27,3.63
26325,2026-07-28,3.63
26326,2026-07-29,3.63
26327,2026-07-30,3.63
26328,2026-07-31,3.63


In [35]:
def build_dataset(ticker: str, price_csv) -> pd.DataFrame:
    prices = pd.read_csv(price_csv)
    prices["Date"] = pd.to_datetime(prices["Date"])
    prices = prices.sort_values("Date").reset_index(drop=True)

    for lag in range(1, LOOKBACK + 1):
        prices[f"close_lag_{lag}"] = prices["Close"].shift(lag)
    prices["rolling_mean_5"] = prices["Close"].rolling(ROLLING_WINDOW).mean()
    prices["rolling_std_5"] = prices["Close"].rolling(ROLLING_WINDOW).std()

    for feature_name, path in FRED_FILES.items():
        econ = load_fred_csv(path, feature_name)
        prices = pd.merge_asof(
            prices.sort_values("Date"), econ,
            left_on="Date", right_on="date", direction="backward",
        ).drop(columns="date")

    prices["next_close"] = prices["Close"].shift(-1)
    prices["target_return"] = (prices["next_close"] - prices["Close"]) / prices["Close"]

    prices = prices.dropna(subset=FEATURE_COLUMNS + ["target_return"]).reset_index(drop=True)
    return prices

In [36]:
df_aapl = build_dataset("AAPL", TICKERS["AAPL"]["csv_path"])
print(df_aapl.shape)
df_aapl[FEATURE_COLUMNS + ["target_return"]].head()

(11497, 18)


,close_lag_1,close_lag_2,close_lag_3,rolling_mean_5,rolling_std_5,interest_rate,inflation_rate,unemployment_rate,gdp_growth,target_return
0,0.115513,0.112723,0.121652,0.119420,0.006023,20.74,86.4,7.2,7315.677,0.061029
1,0.118862,0.115513,0.112723,0.118973,0.005226,20.07,86.4,7.2,7315.677,0.048669
2,0.126116,0.118862,0.115513,0.121094,0.008000,19.75,86.4,7.2,7315.677,0.042199
3,0.132254,0.126116,0.118862,0.126116,0.009220,18.84,86.4,7.2,7315.677,0.052628
4,0.137835,0.132254,0.126116,0.132031,0.010157,16.52,86.4,7.2,7315.677,0.092309


In [37]:
def train_ticker(ticker: str, price_csv):
    df = build_dataset(ticker, price_csv)

    split_idx = int(len(df) * 0.85)
    train, test = df.iloc[:split_idx], df.iloc[split_idx:]

    X_train, y_train = train[FEATURE_COLUMNS], train["target_return"]
    X_test, y_test = test[FEATURE_COLUMNS], test["target_return"]

    model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    print(f"{ticker}: test MAE on next-day return = {mae:.5f}")

    joblib.dump(model, f"{EXPORT_DIR}/{ticker}_econ_model.pkl")
    with open(f"{EXPORT_DIR}/{ticker}_econ_features.json", "w") as f:
        json.dump(FEATURE_COLUMNS, f)

    last_row = df.iloc[-1]
    starting_state = {
        "last_date": df.iloc[-1]["Date"].strftime("%Y-%m-%d"),
        "last_price": float(df.iloc[-1]["Close"]),
        "close_lag_1": float(last_row["close_lag_1"]),
        "close_lag_2": float(last_row["close_lag_2"]),
        "close_lag_3": float(last_row["close_lag_3"]),
        "recent_closes": [float(x) for x in df["Close"].values[-ROLLING_WINDOW:]],
    }
    with open(f"{EXPORT_DIR}/{ticker}_econ_starting_state.json", "w") as f:
        json.dump(starting_state, f)

    return model, df

In [38]:
model_aapl, df_aapl = train_ticker("AAPL", TICKERS["AAPL"]["csv_path"])

AAPL: test MAE on next-day return = 0.01403


In [39]:
model_msft, df_msft = train_ticker("MSFT", TICKERS["MSFT"]["csv_path"])

MSFT: test MAE on next-day return = 0.01449


In [40]:
model_googl, df_googl = train_ticker("GOOGL", TICKERS["GOOGL"]["csv_path"])

GOOGL: test MAE on next-day return = 0.01707


In [41]:
os.listdir(EXPORT_DIR)

['AAPL_econ_model.pkl',
 'AAPL_econ_starting_state.json',
 'GOOGL_econ_model.pkl',
 'MSFT_econ_model.pkl',
 'GOOGL_econ_features.json',
 'GOOGL_econ_starting_state.json',
 'MSFT_econ_starting_state.json',
 'MSFT_econ_features.json',
 'AAPL_econ_features.json']